# Modelo 2: clasificación del tipo de daño (sobre recortes)

Este modelo no mira la imagen completa: recibe un **recorte** ya centrado en
un daño y dice cuál de las 7 clases es. Es un clasificador de imagen común y
corriente (misma idea que `best_model.keras`, el modelo 0), solo que en vez
de responder daño/no-daño responde cuál de 7 tipos.

Para entrenarlo armamos un dataset nuevo: un recorte por cada daño anotado
(no una imagen por foto original). En producción, esos recortes van a salir
de las cajas que encuentra el **modelo 1** (`cnn_damage_location_training.ipynb`);
acá, para entrenar, los recortamos directo de las anotaciones reales.

Al final del notebook conectamos los dos modelos: modelo 1 encuentra las
cajas, este modelo (2) les pone el nombre, y todo junto se dibuja sobre la
imagen original. Más adelante se le puede sumar el modelo 0 al principio de
la cadena (si dice "sin daño", ni se llama a los otros dos).


In [ ]:
import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"


In [ ]:
import subprocess

LOCAL_DATA_DIR = "data_deteccion"
os.makedirs(f"{LOCAL_DATA_DIR}/images", exist_ok=True)
os.makedirs(f"{LOCAL_DATA_DIR}/labels", exist_ok=True)

BUCKET = "s3://humanoid-damage-cnn-834257582282-us-east-1-an/processed/damage_classification"

subprocess.run(["aws", "s3", "sync", f"{BUCKET}/images/", f"{LOCAL_DATA_DIR}/images/", "--quiet"], check=True)
subprocess.run(["aws", "s3", "sync", f"{BUCKET}/labels/", f"{LOCAL_DATA_DIR}/labels/", "--quiet"], check=True)
subprocess.run(["aws", "s3", "cp", f"{BUCKET}/classes.txt", f"{LOCAL_DATA_DIR}/classes.txt", "--quiet"], check=True)

n_images = len(os.listdir(f"{LOCAL_DATA_DIR}/images"))
n_labels = len(os.listdir(f"{LOCAL_DATA_DIR}/labels"))
print(f"Imágenes: {n_images} | Labels: {n_labels}")


In [ ]:
import glob

CLASSES_PATH = f"{LOCAL_DATA_DIR}/classes.txt"
with open(CLASSES_PATH) as f:
    CLASS_NAMES_FULL = [line.strip() for line in f if line.strip()]

# Nos quedamos con estas clases (las que tienen suficientes ejemplos)
CLASS_NAMES = [
    "Diagonal crack",
    "Concrete spalling",
    "Corrosion",
    "Shear crack",
    "Flexural/rocking crack",
    "Debris",
    "Detachment crack",
]
N_CLASSES = len(CLASS_NAMES)

# class_id original -> class_id nuevo (0..N_CLASSES-1). Si no está, es clase excluida.
CLASS_ID_MAP = {
    CLASS_NAMES_FULL.index(nombre): nuevo_id
    for nuevo_id, nombre in enumerate(CLASS_NAMES)
}

print(f"Clases que vamos a usar: {N_CLASSES}")
for nuevo_id, nombre in enumerate(CLASS_NAMES):
    print(f"  {nuevo_id}: {nombre}")


In [ ]:
def get_image_path(label_path):
    img_name = os.path.basename(label_path).replace(".txt", "")
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = f"{LOCAL_DATA_DIR}/images/{img_name}{ext}"
        if os.path.exists(candidate):
            return candidate
    return None

# Una "instancia" acá es UN daño anotado: (imagen de origen, su caja, su clase).
# Un daño por fila, no una imagen por fila.
instancias = []
for label_path in glob.glob(f"{LOCAL_DATA_DIR}/labels/*.txt"):
    image_path = get_image_path(label_path)
    if image_path is None:
        continue
    with open(label_path) as f:
        for line in f:
            partes = line.split()
            if len(partes) != 5:
                continue
            class_id = CLASS_ID_MAP.get(int(partes[0]))
            if class_id is None:
                continue  # clase excluida
            xc, yc, w, h = map(float, partes[1:5])
            instancias.append((image_path, (xc, yc, w, h), class_id))

print(f"Total de recortes de entrenamiento: {len(instancias)}")


In [ ]:
from PIL import Image

IMG_SIZE = (224, 224)
MARGEN = 0.15  # 15% de aire alrededor de la caja real, para dar contexto

def recortar_dano(image_path, box, target_size=IMG_SIZE, margen=MARGEN):
    """Recorta la zona de la caja (con un margen de contexto alrededor) y la
    redimensiona a target_size."""
    xc, yc, w, h = box
    img = Image.open(image_path).convert("RGB")
    img_w, img_h = img.size

    w_con_margen = w * (1 + margen)
    h_con_margen = h * (1 + margen)
    x0 = max(0, (xc - w_con_margen / 2) * img_w)
    y0 = max(0, (yc - h_con_margen / 2) * img_h)
    x1 = min(img_w, (xc + w_con_margen / 2) * img_w)
    y1 = min(img_h, (yc + h_con_margen / 2) * img_h)

    return img.crop((x0, y0, x1, y1)).resize(target_size)


### Cómo se ven los recortes

Verificamos que el recorte + la clase asignada tengan sentido, mirando
algunos al azar.

In [ ]:
#PARA PROBAR
import random
import matplotlib.pyplot as plt

muestras = random.sample(instancias, 6)

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, (image_path, box, class_id) in zip(axes, muestras):
    crop = recortar_dano(image_path, box)
    ax.imshow(crop)
    ax.set_title(CLASS_NAMES[class_id], fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
from collections import Counter

# Split 80/20 por IMAGEN (no por recorte): si dos daños salen de la misma
# foto, tienen que quedar los dos del mismo lado, si no hay data leakage.
imagenes_unicas = sorted(set(inst[0] for inst in instancias))

RANDOM_SEED = 42
TRAIN_RATIO = 0.8
random.seed(RANDOM_SEED)
imagenes_shuffled = list(imagenes_unicas)
random.shuffle(imagenes_shuffled)
n_train = int(len(imagenes_shuffled) * TRAIN_RATIO)
train_imgs = set(imagenes_shuffled[:n_train])
val_imgs = set(imagenes_shuffled[n_train:])

train_instancias = [i for i in instancias if i[0] in train_imgs]
val_instancias = [i for i in instancias if i[0] in val_imgs]

# Pesos por clase (para pasarle a model.fit como class_weight) y sobremuestreo
# leve de las clases más raras en train.
conteo_train = Counter(class_id for _, _, class_id in train_instancias)
max_conteo = max(conteo_train.values())
CLASS_WEIGHTS = {c: max_conteo / conteo_train[c] for c in range(N_CLASSES)}

MAX_REPEATS = 6
train_instancias_balanceadas = []
for inst in train_instancias:
    n_rep = min(MAX_REPEATS, max(1, round(CLASS_WEIGHTS[inst[2]])))
    train_instancias_balanceadas.extend([inst] * n_rep)

print(f"Imágenes: {len(imagenes_unicas)} | Train: {len(train_imgs)} | Validation: {len(val_imgs)}")
print(f"Recortes train (original): {len(train_instancias)} | tras sobremuestreo: {len(train_instancias_balanceadas)}")
print(f"Recortes validation: {len(val_instancias)}")
print("\nPesos por clase:")
for class_id, peso in CLASS_WEIGHTS.items():
    print(f"  {CLASS_NAMES[class_id]}: {peso:.2f}")


In [ ]:
import numpy as np
import tensorflow as tf

BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 1e-4
CHECKPOINT_PATH = "../models/best_type_classifier.keras"
RUN_TRAINING = True
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

def instancias_a_dataset(lista_instancias, batch_size, shuffle=False):
    paths = [inst[0] for inst in lista_instancias]
    boxes = np.array([inst[1] for inst in lista_instancias], dtype=np.float32)
    labels = np.array([inst[2] for inst in lista_instancias], dtype=np.int32)

    def cargar(path, box, label):
        path = path.numpy().decode("utf-8")
        crop = recortar_dano(path, box.numpy())
        return tf.keras.utils.img_to_array(crop), label

    def tf_cargar(path, box, label):
        crop_array, label = tf.py_function(cargar, [path, box, label], [tf.float32, tf.int32])
        crop_array.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
        label.set_shape([])
        return crop_array, label

    ds = tf.data.Dataset.from_tensor_slices((paths, boxes, labels))
    if shuffle:
        ds = ds.shuffle(2000, seed=RANDOM_SEED)
    return ds.map(tf_cargar, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = instancias_a_dataset(train_instancias_balanceadas, BATCH_SIZE, shuffle=True)
val_ds = instancias_a_dataset(val_instancias, BATCH_SIZE)

print("Datasets armados")
print("GPUs detectadas:", tf.config.list_physical_devices("GPU"))


In [ ]:
from tensorflow.keras import layers

# Mismo backbone reutilizado que en el modelo 1 (best_model.keras = modelo 0).
modelo0 = tf.keras.models.load_model("../models/best_model.keras")

backbone_output = None
for layer in modelo0.layers:
    if isinstance(layer, tf.keras.layers.MaxPooling2D):
        backbone_output = layer.output
assert backbone_output is not None, "No se encontró ninguna capa MaxPooling2D en modelo0"

backbone = tf.keras.Model(inputs=modelo0.input, outputs=backbone_output, name="backbone_reutilizado")
backbone.trainable = False  # se descongela en la Fase 2

x = backbone.output
x = layers.GlobalAveragePooling2D(name="tipo_gap")(x)
x = layers.Dense(128, activation="relu", name="tipo_dense")(x)
x = layers.Dropout(0.4)(x)
salida = layers.Dense(N_CLASSES, activation="softmax", name="tipo_output")(x)

modelo_tipo = tf.keras.Model(inputs=backbone.input, outputs=salida, name="modelo_tipo_dano")

# clipnorm como red de seguridad (mismo motivo que en el modelo 1). Acá el
# riesgo de NaN es mucho menor: es un clasificador simple, sin matching ni
# regresión de caja mezclados en la misma loss.
modelo_tipo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
modelo_tipo.summary()


In [ ]:
def crear_callbacks():
    return [
        tf.keras.callbacks.ModelCheckpoint(
            CHECKPOINT_PATH, monitor="val_accuracy", mode="max", save_best_only=True, verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=4, verbose=1,
        ),
        tf.keras.callbacks.TerminateOnNaN(),
    ]

print(f"Callbacks preparados. El mejor clasificador se guardará en: {CHECKPOINT_PATH}")


### Entrenamiento

Dos fases, igual que el modelo 1: warm-up con backbone congelado, después
fine-tuning completo. `class_weight` compensa el desbalance entre clases
directamente en `model.fit`, sin tocar la loss a mano.

In [ ]:
if not RUN_TRAINING:
    print("Entrenamiento preparado pero no iniciado. Cambiar RUN_TRAINING = True para ejecutar model.fit().")
else:
    print("=== Fase 1: warm-up de la cabeza (backbone congelado) ===")
    history_warmup = modelo_tipo.fit(
        train_ds, validation_data=val_ds, epochs=5,
        class_weight=CLASS_WEIGHTS, callbacks=crear_callbacks(), verbose=1,
    )

    print("=== Fase 2: fine-tuning completo (backbone descongelado) ===")
    backbone.trainable = True
    modelo_tipo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10, clipnorm=1.0),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    history = modelo_tipo.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS,
        class_weight=CLASS_WEIGHTS, callbacks=crear_callbacks(), verbose=1,
    )


In [ ]:
if "history" not in globals():
    print("Todavía no hay history: ejecutar primero el entrenamiento.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history.get("loss", []), label="training loss")
    axes[0].plot(history.history.get("val_loss", []), label="validation loss")
    axes[0].set_xlabel("Época"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True)

    axes[1].plot(history.history.get("accuracy", []), label="training accuracy")
    axes[1].plot(history.history.get("val_accuracy", []), label="validation accuracy")
    axes[1].set_xlabel("Época"); axes[1].set_ylabel("Accuracy"); axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.show()


In [ ]:
if os.path.exists(CHECKPOINT_PATH):
    best_type_classifier = tf.keras.models.load_model(CHECKPOINT_PATH)
    print(f"Mejor clasificador cargado desde: {CHECKPOINT_PATH}")
else:
    best_type_classifier = modelo_tipo
    print(f"Aún no existe checkpoint en {CHECKPOINT_PATH}; usando el modelo en memoria.")


### Evaluación

Precision / recall / F1 por clase y matriz de confusión sobre validation.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = best_type_classifier.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(preds.argmax(axis=1))

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Matriz de confusión: tipo de daño")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Conectamos los dos modelos

El modelo 1 (`best_locator.keras`) encuentra las cajas; este notebook las
recorta y el modelo 2 (`best_type_classifier.keras`, recién entrenado) les
pone el nombre. El resultado combinado se dibuja igual que la anotación
real, para poder comparar a simple vista.

In [ ]:
import matplotlib.patches as patches

MAX_BOXES = None  # se completa al cargar el modelo 1 (según su forma de salida)

def detection_loss(y_true, y_pred, lambda_coord=5.0):
    """Firma necesaria para poder deserializar best_locator.keras (fue
    compilado con esta loss en cnn_damage_location_training.ipynb). No hace
    falta que haga el cálculo real para usar el modelo en inferencia."""
    return tf.reduce_mean(tf.square(y_pred[..., 1:5] - y_true[..., 1:5]))

best_locator = tf.keras.models.load_model(
    "../models/best_locator.keras", custom_objects={"detection_loss": detection_loss}
)
MAX_BOXES = best_locator.output_shape[1]
print(f"Localizador cargado. MAX_BOXES = {MAX_BOXES}")


def calcular_iou(box1, box2):
    x1_min, y1_min = box1[0] - box1[2] / 2, box1[1] - box1[3] / 2
    x1_max, y1_max = box1[0] + box1[2] / 2, box1[1] + box1[3] / 2
    x2_min, y2_min = box2[0] - box2[2] / 2, box2[1] - box2[3] / 2
    x2_max, y2_max = box2[0] + box2[2] / 2, box2[1] + box2[3] / 2
    inter_xmin, inter_ymin = max(x1_min, x2_min), max(y1_min, y2_min)
    inter_xmax, inter_ymax = min(x1_max, x2_max), min(y1_max, y2_max)
    inter_area = max(0, inter_xmax - inter_xmin) * max(0, inter_ymax - inter_ymin)
    union_area = box1[2] * box1[3] + box2[2] * box2[3] - inter_area
    return inter_area / union_area if union_area > 0 else 0.0


def non_max_suppression(predicciones, umbral_iou_nms=0.4):
    predicciones = sorted(predicciones, key=lambda p: p[4], reverse=True)
    resultado = []
    while predicciones:
        mejor = predicciones.pop(0)
        resultado.append(mejor)
        predicciones = [p for p in predicciones if calcular_iou(mejor[:4], p[:4]) < umbral_iou_nms]
    return resultado


def localizar_danios(image_path, umbral_objectness=0.3, umbral_iou_nms=0.4, img_size=(224, 224)):
    """Corre el modelo 1 y devuelve las cajas encontradas (xc, yc, w, h, objectness)."""
    img = Image.open(image_path).convert("RGB").resize(img_size)
    img_array = tf.keras.utils.img_to_array(img)
    pred = best_locator(tf.expand_dims(img_array, 0), training=False)[0]

    predicciones = []
    for slot in range(pred.shape[0]):
        objectness = tf.sigmoid(pred[slot, 0]).numpy()
        if objectness < umbral_objectness:
            continue
        xc, yc, w, h = pred[slot, 1:5].numpy()
        predicciones.append((xc, yc, w, h, objectness))

    return non_max_suppression(predicciones, umbral_iou_nms)


def detectar_y_clasificar(image_path, umbral_objectness=0.3):
    """Pipeline completo: modelo 1 encuentra las cajas, modelo 2 clasifica
    cada una. Devuelve (xc, yc, w, h, class_id, objectness, confianza_clase)."""
    cajas = localizar_danios(image_path, umbral_objectness=umbral_objectness)
    resultados = []
    for xc, yc, w, h, objectness in cajas:
        crop = recortar_dano(image_path, (xc, yc, w, h))
        crop_array = np.expand_dims(tf.keras.utils.img_to_array(crop), 0)
        probs = best_type_classifier(crop_array, training=False)[0].numpy()
        class_id = int(np.argmax(probs))
        resultados.append((xc, yc, w, h, class_id, objectness, float(probs[class_id])))
    return resultados


def plot_pipeline_on_image(image_path, ax, umbral_objectness=0.3):
    resultados = detectar_y_clasificar(image_path, umbral_objectness=umbral_objectness)
    img = Image.open(image_path).convert("RGB")
    img_w, img_h = img.size
    ax.imshow(img)
    for xc, yc, w, h, class_id, objectness, confianza in resultados:
        box_w, box_h = w * img_w, h * img_h
        x0, y0 = xc * img_w - box_w / 2, yc * img_h - box_h / 2
        rect = patches.Rectangle((x0, y0), box_w, box_h, linewidth=2, edgecolor="red", facecolor="none")
        ax.add_patch(rect)
        ax.text(x0, y0 - 4, f"{CLASS_NAMES[class_id]} ({confianza:.2f})",
                color="red", fontsize=7, weight="bold")
    ax.set_title(f"Pipeline (modelo 1 + modelo 2): {os.path.basename(image_path)}", fontsize=9)
    ax.axis("off")


In [ ]:
def plot_real_box_on_image(image_path, ax):
    """Anotación real (con clase), para comparar contra el pipeline."""
    label_path = image_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    img = Image.open(image_path).convert("RGB")
    img_w, img_h = img.size
    ax.imshow(img)

    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                partes = line.split()
                if len(partes) != 5:
                    continue
                class_id = CLASS_ID_MAP.get(int(partes[0]))
                if class_id is None:
                    continue
                xc, yc, w, h = map(float, partes[1:5])
                box_w, box_h = w * img_w, h * img_h
                x0, y0 = xc * img_w - box_w / 2, yc * img_h - box_h / 2
                rect = patches.Rectangle((x0, y0), box_w, box_h, linewidth=2, edgecolor="red", facecolor="none")
                ax.add_patch(rect)
                ax.text(x0, y0 - 4, CLASS_NAMES[class_id], color="red", fontsize=7, weight="bold")

    ax.set_title(f"Real: {os.path.basename(image_path)}", fontsize=9)
    ax.axis("off")


# Elegimos una imagen random de validation (recortes) que tenga daño real
imagenes_val_con_dano = sorted(val_imgs)
imagen_demo = random.choice(imagenes_val_con_dano)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
plot_real_box_on_image(imagen_demo, axes[0])
plot_pipeline_on_image(imagen_demo, axes[1], umbral_objectness=0.3)
plt.tight_layout()
plt.show()


### A futuro: sumar el modelo 0

`best_model.keras` (modelo 0) ya dice si una imagen tiene daño o no. Se
puede anteponer a esta cadena: si modelo 0 dice "sin daño", ni se llama a
`detectar_y_clasificar` — se ahorra cómputo y se evita que el modelo 1
alucine una caja en una imagen sin ningún daño real. No se implementa acá
todavía, pero la función de abajo muestra la idea.

In [ ]:
def pipeline_completo(image_path, umbral_dano=0.5, umbral_objectness=0.3):
    modelo0 = tf.keras.models.load_model("../models/best_model.keras")
    img = tf.keras.utils.load_img(image_path, target_size=(224, 224))
    img_array = np.expand_dims(tf.keras.utils.img_to_array(img), 0)
    prob_no_danio = modelo0.predict(img_array, verbose=0)[0][0]

    if prob_no_danio >= umbral_dano:
        return []  # modelo 0 dice que no hay daño: no llamamos a los otros dos

    return detectar_y_clasificar(image_path, umbral_objectness=umbral_objectness)
